# Session 03 · Guided lab — Pretrained Models and Transfer Learning
**Desarrollo y Evaluación de Modelos Propios** · Especialización en IA Generativa y Desarrollo de Negocios · Universidad del Rosario · Rosario GSB

[![Open in Kaggle](https://kaggle.com/static/images/open-in-kaggle.svg)](https://kaggle.com/kernels/welcome?src=https://github.com/juccaicedoac03/Desarrollo-Evaluacion-Modelos/blob/main/sessions/03-transfer-learning/lab.ipynb)
[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/juccaicedoac03/Desarrollo-Evaluacion-Modelos/blob/main/sessions/03-transfer-learning/lab.ipynb)

In the slides we saw that a pretrained model can be reused in several ways: prompting it with no examples (zero-shot), prompting it with a few solved examples (few-shot), or freezing it and training a small classifier on its vectors (feature extraction). Today you try all of them on the **same news test set** and compare accuracy, labelled data and latency, the three numbers behind any transfer-learning decision.

**Learning objectives**
1. Compare how an English-centric and a multilingual tokenizer split English and Spanish text, and translate the difference into cost.
2. Probe what BERT learned during pretraining with its masked-language-modelling head.
3. Build a classifier with frozen sentence embeddings (`sentence-transformers/all-MiniLM-L6-v2`) and logistic regression.
4. Classify the same texts zero-shot and few-shot with a small chat LLM (`Qwen/Qwen2.5-0.5B-Instruct`).
5. Read a comparison table of accuracy, labelled examples and latency like a product owner would.

**Time plan (60 min)**

| Part | Topic | Time |
|---|---|---|
| 1 | Tokenizers: BERT vs GPT-2 (and Qwen2.5) on English/Spanish sentences | 12' |
| 2 | Fill-mask with BERT: knowledge, bias and special tokens | 8' |
| 3 | Feature extraction: embeddings + logistic regression on AG News | 15' |
| 4 | Zero-shot and few-shot classification with Qwen2.5-0.5B-Instruct | 15' |
| 5 | Comparison table and wrap-up (project groups) | 10' |

**Kaggle checklist**
- *Settings → Internet → On* (models and data are downloaded from Hugging Face).
- *Settings → Accelerator → GPU T4 x2* is **recommended** for Part 4; the notebook also runs on CPU, Part 4 is just slower.
- Run the cells in order, from top to bottom. If you restart the session, run the setup cell again.

In [ ]:
# --- Course setup (do not edit) ---
import os, sys, random, subprocess, importlib
FAST_DEV_RUN = os.environ.get("COURSE_FAST_DEV_RUN") == "1"  # used by the course smoke test
IN_KAGGLE = "KAGGLE_KERNEL_RUN_TYPE" in os.environ
IN_COLAB = "google.colab" in sys.modules

def ensure(package, import_name=None):
    """Install a package only if it is missing (Kaggle already ships most of them)."""
    try:
        importlib.import_module(import_name or package)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", package], check=True)

import numpy as np
import torch
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE} | Kaggle: {IN_KAGGLE} | Colab: {IN_COLAB} | Fast dev run: {FAST_DEV_RUN}")
# --- end of course setup ---
ensure("transformers")
ensure("datasets")
ensure("scikit-learn", "sklearn")
ensure("pandas")
ensure("matplotlib")

Sizes used in this lab. `FAST_DEV_RUN` is only switched on by the course's automatic test: it runs the same code with tiny numbers. In class you use the full values.

In [ ]:
import re, time, gc
import pandas as pd
import matplotlib.pyplot as plt

K_PER_CLASS = 8 if FAST_DEV_RUN else 64        # labelled training examples per class (Part 3)
N_TEST = 40 if FAST_DEV_RUN else 200           # test articles for the embeddings classifier
N_TEST_LLM = 8 if FAST_DEV_RUN else 40         # test articles for the LLM (slower, so fewer)
SHOTS_PER_CLASS = 1                            # solved examples per class in the few-shot prompt
MAX_NEW_TOKENS = 4 if FAST_DEV_RUN else 6      # the LLM only needs to write one label
pd.set_option("display.max_colwidth", 80)
print(f"k per class = {K_PER_CLASS} · test = {N_TEST} · LLM test = {N_TEST_LLM} · shots per class = {SHOTS_PER_CLASS}")

## Part 1 — Tokenizers: how many pieces does your text cost?

A model never sees words: it sees **tokens** from a fixed vocabulary. We compare three tokenizers on eight sentences that say the same thing in English and in Spanish:

| Tokenizer | Algorithm | Vocabulary | Trained mostly on |
|---|---|---|---|
| `google-bert/bert-base-uncased` | WordPiece (`##` = continuation) | 30,522 | English |
| `openai-community/gpt2` | byte-level BPE (`Ġ` = leading space) | 50,257 | English |
| `Qwen/Qwen2.5-0.5B-Instruct` | byte-level BPE | about 150K | many languages |

Downloading a tokenizer is fast: it is a vocabulary file, not the model.

In [ ]:
from transformers import AutoTokenizer

TOKENIZER_NAMES = {
    "BERT": "google-bert/bert-base-uncased",
    "GPT-2": "openai-community/gpt2",
    "Qwen2.5": "Qwen/Qwen2.5-0.5B-Instruct",
}
tokenizers = {short: AutoTokenizer.from_pretrained(name) for short, name in TOKENIZER_NAMES.items()}

# Eight parallel sentences (English, Spanish) from customer service and business news.
PAIRS = [
    ("Your card payment was declined because of insufficient funds.",
     "El pago con tu tarjeta fue rechazado por fondos insuficientes."),
    ("We will refund the full amount within five business days.",
     "Reembolsaremos el valor total en un plazo de cinco días hábiles."),
    ("The customer wants to cancel the subscription and speak with a manager.",
     "El cliente quiere cancelar la suscripción y hablar con un supervisor."),
    ("Please update your address before the next delivery.",
     "Por favor, actualiza tu dirección antes de la próxima entrega."),
    ("Interest rates remained unchanged after the central bank meeting.",
     "Las tasas de interés se mantuvieron sin cambios tras la reunión del banco central."),
    ("Thank you for your patience while we investigate the issue.",
     "Gracias por tu paciencia mientras investigamos el problema."),
    ("The new mobile app lets you transfer money in seconds.",
     "La nueva aplicación móvil te permite transferir dinero en segundos."),
    ("Our technicians will visit your home tomorrow morning.",
     "Nuestros técnicos visitarán tu hogar mañana por la mañana."),
]

en, es = PAIRS[1]
for short, tok in tokenizers.items():
    print(f"{short:8s} EN ({len(tok.tokenize(en))}): {tok.tokenize(en)}")
    print(f"{'':8s} ES ({len(tok.tokenize(es))}): {tok.tokenize(es)}\n")

What to notice:
- **BERT** marks word continuations with `##`. This checkpoint is *uncased*: it lowercases the text and also strips accents (`días` → `dia ##s`), so it cannot tell *papa* from *papá*.
- **GPT-2** works on bytes. `Ġ` means "this piece starts with a space"; accented letters take two bytes in UTF-8, so `í` shows up as an odd piece such as `ÃŃ`.
- **Qwen2.5** was built for many languages, so frequent Spanish words stay whole more often.

Now count tokens for all eight pairs.

In [ ]:
rows = []
for en, es in PAIRS:
    row = {"English": en, "EN words": len(en.split()), "ES words": len(es.split())}
    for short, tok in tokenizers.items():
        row[f"{short} EN"] = len(tok.tokenize(en))
        row[f"{short} ES"] = len(tok.tokenize(es))
    rows.append(row)
tokens_df = pd.DataFrame(rows)
display(tokens_df.drop(columns="English"))

total_en_words = tokens_df["EN words"].sum()
summary = []
for short in tokenizers:
    en_tokens, es_tokens = tokens_df[f"{short} EN"].sum(), tokens_df[f"{short} ES"].sum()
    summary.append({"tokenizer": short,
                    "EN tokens": en_tokens, "ES tokens": es_tokens,
                    "tokens per EN word": round(en_tokens / total_en_words, 2),
                    "ES/EN ratio": round(es_tokens / en_tokens, 2)})
ratios_df = pd.DataFrame(summary).set_index("tokenizer")
ratios_df

### From tokens to money

APIs bill per token, and each output token is one more decoding step (latency). The cell below turns the ratios into a monthly bill for the **same content** in each language. The price is a placeholder, not a real quote: replace it with your provider's current price for the model you would use.

In [ ]:
WORDS_PER_REQUEST = 400        # English words of content per request (prompt + answer)
REQUESTS_PER_MONTH = 100_000
PRICE_PER_1K_TOKENS = 0.001    # USD — illustrative placeholder, look up a current price

cost_rows = []
for short, r in ratios_df.iterrows():
    tokens_en = WORDS_PER_REQUEST * r["tokens per EN word"] * REQUESTS_PER_MONTH
    tokens_es = tokens_en * r["ES/EN ratio"]
    cost_rows.append({"tokenizer": short,
                      "EN tokens / month (M)": round(tokens_en / 1e6, 1),
                      "ES tokens / month (M)": round(tokens_es / 1e6, 1),
                      "EN cost (USD)": round(tokens_en / 1000 * PRICE_PER_1K_TOKENS, 2),
                      "ES cost (USD)": round(tokens_es / 1000 * PRICE_PER_1K_TOKENS, 2),
                      "Spanish premium": f"+{(r['ES/EN ratio'] - 1) * 100:.0f}%"})
pd.DataFrame(cost_rows).set_index("tokenizer")

**Special tokens.** BERT adds `[CLS]` at the start and `[SEP]` at the end of every input. The final vector of `[CLS]` is what BERT's classification head reads.

In [ ]:
bert_tok = tokenizers["BERT"]
encoded = bert_tok("the refund arrived")
print("ids:   ", encoded["input_ids"])
print("tokens:", bert_tok.convert_ids_to_tokens(encoded["input_ids"]))

**✋ Checkpoint (Zoom chat):** which tokenizer would you choose to estimate the cost of a Spanish chatbot, and why is the ES/EN ratio more useful than "tokens per word"?

**🧪 Try it:** add two sentences from your own work (English and Spanish versions) to `PAIRS` and rerun the table. Does your ratio look like ours?

<details><summary>Show a solution</summary>

```python
PAIRS.append(("Your claim has been approved and the payment is on its way.",
              "Tu reclamación fue aprobada y el pago ya está en camino."))
# then rerun the counting cell: ratios move a little with every new sentence,
# which is why you should measure on a sample of your *real* texts.
```
</details>

## Part 2 — Fill-mask: what did BERT learn during pretraining?

BERT was pretrained to recover masked tokens using the words on both sides (masked language modelling). The `fill-mask` pipeline shows its top guesses for `[MASK]`. No labels were ever given for these tasks: everything below comes from self-supervision.

In [ ]:
from transformers import pipeline

fill_mask = pipeline("fill-mask", model="google-bert/bert-base-uncased", device=DEVICE)

def show_fill(sentence, top_k=5):
    preds = fill_mask(sentence, top_k=top_k)
    guesses = " · ".join(f"{p['token_str']} {p['score']:.2f}" for p in preds)
    print(f"{sentence}\n   → {guesses}\n")

for s in ["The bank approved my [MASK] application.",
          "Paris is the [MASK] of France.",
          "The customer asked for a [MASK] because the product arrived broken."]:
    show_fill(s)

Pretraining also absorbs the **biases** of its corpus, and an English model knows nothing about Spanish:

In [ ]:
for s in ["The nurse said [MASK] would be back in five minutes.",
          "The engineer said [MASK] would be back in five minutes.",
          "El banco aprobó mi [MASK] de crédito."]:
    show_fill(s)

# free memory before loading the next models
del fill_mask
gc.collect()

**✋ Checkpoint (Zoom chat):** the nurse/engineer pair changes only one word. What would this bias mean for a CV-screening or customer-routing model built on top of BERT? (We come back to bias audits in Session 10.)

**🧪 Try it:** write a `[MASK]` sentence about your industry. Does BERT know its vocabulary?

<details><summary>Show a solution</summary>

```python
fill_mask = pipeline("fill-mask", model="google-bert/bert-base-uncased", device=DEVICE)
show_fill("The insurance company rejected the [MASK] because the policy had expired.")
# "claim" should be near the top. For Spanish text you would need a Spanish model
# such as BETO (dccuchile/bert-base-spanish-wwm-cased), which uses [MASK] the same way.
```
</details>

## Part 3 — Feature extraction: frozen embeddings + logistic regression

**Data.** [AG News](https://huggingface.co/datasets/fancyzhx/ag_news): news titles and descriptions in four topics (World, Sports, Business, Sci/Tech), 120,000 for training and 7,600 for testing. The raw text contains artefacts from its web source (`#39;` for an apostrophe, `quot;`, backslashes), so we clean it first: a small reminder of Session 02's preprocessing.

In [ ]:
from datasets import load_dataset

ag_news = load_dataset("fancyzhx/ag_news")
LABELS = ag_news["train"].features["label"].names   # ['World', 'Sports', 'Business', 'Sci/Tech']
N_CLASSES = len(LABELS)

def _entity(match):
    code = int(match.group(1))
    return chr(code) if code < 128 else " "          # '#39;' → "'", '#36;' → '$'

def clean(text):
    """Undo AG News artefacts: numeric entities, 'quot;', backslashes and HTML tags."""
    text = re.sub(r"#(\d+);", _entity, text)
    text = text.replace("quot;", '"').replace("\\", " ")
    text = re.sub(r"&lt;.*?&gt;", " ", text)         # escaped HTML tags such as &lt;b&gt;
    return re.sub(r"\s+", " ", text).strip()

def sample_per_class(split, k, seed):
    """k random examples of each class (class-blocked order). Same seed → nested samples."""
    rng = np.random.default_rng(seed)
    labels = np.array(split["label"])
    idx = np.concatenate([rng.permutation(np.where(labels == c)[0])[:k] for c in range(N_CLASSES)])
    subset = split.select(idx.tolist())
    return [clean(t) for t in subset["text"]], np.array(subset["label"])

train_texts, y_train = sample_per_class(ag_news["train"], K_PER_CLASS, seed=SEED)
test_split = ag_news["test"].shuffle(seed=SEED).select(range(N_TEST))
test_texts, y_test = [clean(t) for t in test_split["text"]], np.array(test_split["label"])

print(f"Train: {len(train_texts)} articles ({K_PER_CLASS} per class) · Test: {len(test_texts)} articles")
print("Test label counts:", dict(zip(LABELS, np.bincount(y_test, minlength=N_CLASSES))))
print("\nRaw:  ", ag_news["test"][1]["text"][:150])
print("Clean:", clean(ag_news["test"][1]["text"])[:150])

**The encoder.** `all-MiniLM-L6-v2` is a small 6-layer Transformer encoder fine-tuned so that texts with similar meaning get similar vectors (Sentence-BERT approach, Reimers & Gurevych, 2019). We load it with plain `transformers` and do **mean pooling** ourselves: average the token vectors, ignoring padding, then normalise to length 1. That is all the `sentence-transformers` library does for this model.

In [ ]:
from transformers import AutoModel
import torch.nn.functional as F

EMB_NAME = "sentence-transformers/all-MiniLM-L6-v2"
emb_tok = AutoTokenizer.from_pretrained(EMB_NAME)
encoder = AutoModel.from_pretrained(EMB_NAME).to(DEVICE).eval()
print(f"{EMB_NAME}: {sum(p.numel() for p in encoder.parameters()) / 1e6:.1f} M parameters")

@torch.no_grad()
def embed(texts, batch_size=64):
    """One 384-number vector per text: mean of the token vectors, L2-normalised."""
    vectors = []
    for i in range(0, len(texts), batch_size):
        enc = emb_tok(texts[i:i + batch_size], padding=True, truncation=True,
                      max_length=128, return_tensors="pt").to(DEVICE)
        hidden = encoder(**enc).last_hidden_state                 # (batch, tokens, 384)
        mask = enc["attention_mask"].unsqueeze(-1).float()        # 1 for real tokens, 0 for padding
        mean = (hidden * mask).sum(dim=1) / mask.sum(dim=1).clamp(min=1e-9)
        vectors.append(F.normalize(mean, dim=1).cpu())
    return torch.cat(vectors).numpy()

start = time.perf_counter()
X_train = embed(train_texts)
train_embed_s = time.perf_counter() - start
start = time.perf_counter()
X_test = embed(test_texts)
test_embed_s = time.perf_counter() - start
print("X_train:", X_train.shape, "· X_test:", X_test.shape)
print(f"Embedding time: {1000 * test_embed_s / len(test_texts):.1f} ms per test article")

Before training anything, look at the vectors: a 2-D projection (PCA) of the test embeddings, coloured by the true topic. If topics already form clusters, a simple linear classifier will do well.

In [ ]:
from sklearn.decomposition import PCA

xy = PCA(n_components=2, random_state=SEED).fit_transform(X_test)
fig, ax = plt.subplots(figsize=(6, 4.5))
for c, name in enumerate(LABELS):
    ax.scatter(xy[y_test == c, 0], xy[y_test == c, 1], s=18, label=name, alpha=0.8)
ax.set_title("Test articles: MiniLM embeddings projected to 2-D (PCA)")
ax.set_xlabel("PC 1"); ax.set_ylabel("PC 2"); ax.legend()
plt.show()

**The head.** Logistic regression on the frozen vectors: the encoder is not changed at all, so training takes seconds on a CPU.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix

start = time.perf_counter()
clf = LogisticRegression(max_iter=1000).fit(X_train, y_train)
lr_train_s = time.perf_counter() - start
start = time.perf_counter()
lr_pred = clf.predict(X_test)
lr_predict_s = time.perf_counter() - start

lr_acc = accuracy_score(y_test, lr_pred)
lr_ms = 1000 * (test_embed_s + lr_predict_s) / len(test_texts)
print(f"Accuracy with {K_PER_CLASS} labelled examples per class: {lr_acc:.3f}")
print(f"Training time: {lr_train_s:.2f} s · inference: {lr_ms:.1f} ms per article (embedding + prediction)")
pd.DataFrame(confusion_matrix(y_test, lr_pred, labels=range(N_CLASSES)),
             index=[f"true {l}" for l in LABELS], columns=[f"pred {l}" for l in LABELS])

**✋ Checkpoint (Zoom chat):** which two topics does the classifier confuse most? Is that a model problem or a data problem (think of a technology company's quarterly results)?

**🧪 Try it:** change `K_PER_CLASS` to 16 and to 256 (first cell of this lab) and rerun Part 3. How much accuracy does each 4× increase buy? You will measure this curve carefully in the challenge.

<details><summary>Show a solution</summary>

```python
for k in [16, 64, 256]:
    texts_k, y_k = sample_per_class(ag_news["train"], k, seed=SEED)
    acc = accuracy_score(y_test, LogisticRegression(max_iter=1000).fit(embed(texts_k), y_k).predict(X_test))
    print(k, round(acc, 3))
# Expect fast gains at first and then a flattening curve: the frozen encoder sets a ceiling.
```

And to check that our mean pooling matches the `sentence-transformers` library (if it is installed):

```python
from sentence_transformers import SentenceTransformer
st = SentenceTransformer(EMB_NAME, device=DEVICE)
ours, theirs = embed(test_texts[:5]), st.encode(test_texts[:5], normalize_embeddings=True)
print((ours * theirs).sum(axis=1))   # cosine similarities ≈ 1.0
```
</details>

## Part 4 — Zero-shot and few-shot classification with a small LLM

Now the other end of the spectrum: no classifier training at all. We **prompt** `Qwen/Qwen2.5-0.5B-Instruct`, a multilingual chat model with about 0.5 billion parameters (Apache-2.0 licence), and ask it for the label.

- **Zero-shot:** only an instruction.
- **Few-shot (in-context learning):** the same instruction plus solved examples, written as previous turns of the conversation. The weights do not change: the examples only live in this prompt.

The model answers in free text, so we **parse** the answer into one of the four labels and count anything else as `unknown`.

In [ ]:
from transformers import AutoModelForCausalLM

LLM_NAME = "Qwen/Qwen2.5-0.5B-Instruct"
llm_tok = tokenizers["Qwen2.5"]                       # same tokenizer we loaded in Part 1
llm = AutoModelForCausalLM.from_pretrained(LLM_NAME).to(DEVICE).eval()
print(f"{LLM_NAME}: {sum(p.numel() for p in llm.parameters()) / 1e6:.0f} M parameters on {DEVICE}")

SYSTEM_PROMPT = ("You are a news topic classifier. Read the article and answer with exactly one label "
                 "from this list: World, Sports, Business, Sci/Tech. Answer with the label only.")

def shorten(text, max_words):
    words = text.split()
    return " ".join(words[:max_words]) + (" …" if len(words) > max_words else "")

def build_messages(text, examples=()):
    """System instruction + optional solved examples (user/assistant turns) + the new article."""
    messages = [{"role": "system", "content": SYSTEM_PROMPT}]
    for ex_text, ex_label in examples:
        messages.append({"role": "user", "content": "Article: " + shorten(ex_text, 40)})
        messages.append({"role": "assistant", "content": LABELS[ex_label]})
    messages.append({"role": "user", "content": "Article: " + shorten(text, 60)})
    return messages

LABEL_PATTERNS = {"Sci/Tech": r"sci|tech", "Sports": r"sport", "Business": r"business", "World": r"world"}

def parse_label(answer):
    """Map the model's free text to a label: the pattern that appears first wins; else 'unknown'."""
    answer = answer.lower()
    best_label, best_pos = "unknown", len(answer) + 1
    for label, pattern in LABEL_PATTERNS.items():
        match = re.search(pattern, answer)
        if match and match.start() < best_pos:
            best_label, best_pos = label, match.start()
    return best_label

print(parse_label("Sci/Tech"), "|", parse_label("The answer is: Sports."), "|", parse_label("I cannot tell"))

Let us look at the exact text the model receives. The **chat template** wraps each message in the special tokens Qwen was trained with; getting this format wrong is a common source of poor results.

In [ ]:
few_shot_examples = list(zip(*sample_per_class(ag_news["train"], SHOTS_PER_CLASS, seed=SEED + 1)))
random.Random(SEED).shuffle(few_shot_examples)       # mix the classes so the order is not always World → Sci/Tech
print(llm_tok.apply_chat_template(build_messages(test_texts[0], few_shot_examples[:2]),
                                  add_generation_prompt=True, tokenize=False))

In [ ]:
@torch.no_grad()
def llm_classify(texts, examples=()):
    """Greedy-decode a short answer for each text. Returns labels, raw answers, seconds and prompt lengths."""
    labels, answers, seconds, prompt_tokens = [], [], [], []
    for text in texts:
        inputs = llm_tok.apply_chat_template(build_messages(text, examples), add_generation_prompt=True,
                                             return_tensors="pt", return_dict=True).to(DEVICE)
        start = time.perf_counter()
        output = llm.generate(**inputs, max_new_tokens=MAX_NEW_TOKENS, do_sample=False,
                              pad_token_id=llm_tok.eos_token_id)
        seconds.append(time.perf_counter() - start)
        answer = llm_tok.decode(output[0, inputs["input_ids"].shape[1]:], skip_special_tokens=True)
        answers.append(answer.strip())
        labels.append(parse_label(answer))
        prompt_tokens.append(inputs["input_ids"].shape[1])
    return labels, answers, seconds, prompt_tokens

llm_texts, llm_y = test_texts[:N_TEST_LLM], y_test[:N_TEST_LLM]
zs_labels, zs_answers, zs_seconds, zs_tokens = llm_classify(llm_texts)
zs_acc = np.mean([p == LABELS[y] for p, y in zip(zs_labels, llm_y)])
print(f"Zero-shot accuracy on {N_TEST_LLM} articles: {zs_acc:.3f} · unknown: {zs_labels.count('unknown')} · "
      f"{1000 * np.mean(zs_seconds):.0f} ms per article · {np.mean(zs_tokens):.0f} prompt tokens")

In [ ]:
fs_labels, fs_answers, fs_seconds, fs_tokens = llm_classify(llm_texts, few_shot_examples)
fs_acc = np.mean([p == LABELS[y] for p, y in zip(fs_labels, llm_y)])
print(f"Few-shot ({SHOTS_PER_CLASS} per class) accuracy: {fs_acc:.3f} · unknown: {fs_labels.count('unknown')} · "
      f"{1000 * np.mean(fs_seconds):.0f} ms per article · {np.mean(fs_tokens):.0f} prompt tokens")

pd.DataFrame({"article": [shorten(t, 12) for t in llm_texts], "true": [LABELS[y] for y in llm_y],
              "zero-shot raw": zs_answers, "few-shot raw": fs_answers, "few-shot label": fs_labels}).head(10)

**✋ Checkpoint (Zoom chat):** do the raw answers always look like a clean label? Which label does the model over-use? How many prompt tokens did the examples add, and what does that mean for the bill of an API?

**🧪 Try it:** translate the system prompt into Spanish (keep the four labels in English) and rerun zero-shot. Does accuracy change?

<details><summary>Show a solution</summary>

```python
SYSTEM_PROMPT = ("Eres un clasificador de noticias. Lee el artículo y responde con una sola etiqueta "
                 "de esta lista: World, Sports, Business, Sci/Tech. Responde solo con la etiqueta.")
zs_labels_es, *_ = llm_classify(llm_texts)
print(np.mean([p == LABELS[y] for p, y in zip(zs_labels_es, llm_y)]))
# Small models are sensitive to wording: always evaluate a prompt change on the same test set.
```
</details>

## Part 5 — The comparison table

A fair comparison uses the **same test articles** for every strategy. The LLM only saw the first `N_TEST_LLM` articles, so we score the embedding classifiers on that same subset. We also train logistic regression on **exactly the same few examples** the LLM saw in its prompt.

In [ ]:
lr_subset_acc = accuracy_score(llm_y, lr_pred[:N_TEST_LLM])
few_texts = [t for t, _ in few_shot_examples]
few_y = np.array([y for _, y in few_shot_examples])
lr_few = LogisticRegression(max_iter=1000).fit(embed(few_texts), few_y)
lr_few_acc = accuracy_score(llm_y, lr_few.predict(X_test[:N_TEST_LLM]))

comparison = pd.DataFrame([
    {"strategy": "Zero-shot LLM (Qwen2.5-0.5B)", "labelled examples": 0,
     "accuracy": zs_acc, "ms per article": 1000 * np.mean(zs_seconds)},
    {"strategy": f"Few-shot LLM ({SHOTS_PER_CLASS}/class in the prompt)", "labelled examples": len(few_shot_examples),
     "accuracy": fs_acc, "ms per article": 1000 * np.mean(fs_seconds)},
    {"strategy": f"Embeddings + LR (same {len(few_shot_examples)} examples)", "labelled examples": len(few_shot_examples),
     "accuracy": lr_few_acc, "ms per article": lr_ms},
    {"strategy": f"Embeddings + LR ({K_PER_CLASS}/class)", "labelled examples": len(train_texts),
     "accuracy": lr_subset_acc, "ms per article": lr_ms},
]).set_index("strategy")
print(f"Same {N_TEST_LLM} test articles for every row · device: {DEVICE}")
comparison.round({"accuracy": 3, "ms per article": 1})

**How to read it (business lens).**
- **Accuracy** on a few dozen articles is noisy: with 40 articles, one article is 2.5 points. Look for large gaps, not decimals.
- **Labelled examples** are the hidden cost: someone has to label them, and more labels usually buy accuracy with diminishing returns.
- **Latency** decides where a model can run: the LLM writes its answer token by token, while the classifier needs one forward pass of a small encoder plus a dot product.
- The LLM needed **no training**, which makes it the fastest *project* to start; the classifier is the cheapest *system* to run at volume.

**✋ Checkpoint (Zoom chat):** Andes Bank must classify 2 million short texts a month on its own servers. Which row would you start from, and which number worries you most?

**🧪 Try it:** set `SHOTS_PER_CLASS = 2` (first cell), rerun Part 4 and Part 5. Did the extra examples help the LLM as much as they help logistic regression?

<details><summary>Show a solution</summary>

```python
few_shot_examples = list(zip(*sample_per_class(ag_news["train"], 2, seed=SEED + 1)))
random.Random(SEED).shuffle(few_shot_examples)
fs2_labels, _, fs2_seconds, fs2_tokens = llm_classify(llm_texts, few_shot_examples)
print(np.mean([p == LABELS[y] for p, y in zip(fs2_labels, llm_y)]), np.mean(fs2_tokens))
# More shots make the prompt longer (slower, more tokens) and do not always help a 0.5B model;
# logistic regression usually improves steadily with more labels.
```
</details>

## Wrap-up

**Key takeaways**
- Tokenizers trained mostly on English split Spanish into roughly twice as many tokens; a multilingual tokenizer narrows the gap. Tokens are cost, context and latency.
- Pretraining gives BERT real knowledge (and real biases) without a single label.
- Frozen embeddings + logistic regression is a strong, cheap and fast baseline once you have a few dozen labels per class.
- Zero-/few-shot prompting needs no training but pays in latency, prompt tokens and parsing failures; few-shot examples do not change the weights.

**Next: the Session Challenge.** You will measure a full learning curve with your own `k_train`, run few-shot with your own `n_shots` on your personal test slice, and recommend a strategy for a business case.

**Final project — milestone H1 (today).** Form your group of 3 (mix business and technical profiles). One member uploads the group registration to e-Aulas as `PF_H1_<codigo>` (their own student code) by 23:59 today. The template block and the suggested roles are in the [final-project brief](https://github.com/juccaicedoac03/Desarrollo-Evaluacion-Modelos/blob/main/proyecto-final/README.md), section 3.